In [1]:
import sys, torch, os
print("Python", sys.version.split()[0], "| Torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || true
!free -g | head -2

Python 3.11.13 | Torch 2.6.0+cu124 | CUDA True
GPU: NVIDIA A100-SXM4-40GB 42 GB
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
               total        used        free      shared  buff/cache   available
Mem:              83           3          75           0           4          79


In [8]:
!pip install -q onnx onnxruntime jiwer unidecode h5py pyyaml flashlight-text huggingface_hub pandas praat-textgrids
from torchaudio.models.decoder import ctc_decoder
print("torchaudio ctc_decoder import OK")

torchaudio ctc_decoder import OK


In [9]:
from google.colab import drive
drive.mount("/content/drive")
import os, sys, glob, shutil
# ===================== EDIT THESE =====================
DRIVE_PROJECT = "/content/drive/MyDrive/KD/silent_speech"        # your project folder on Drive
GADDY_CKPT    = f"{DRIVE_PROJECT}/gaddy_recognition_model.pt"  # your 54M Gaddy checkpoint (teacher)
# =====================================================
REPO_DIR = "/content/silent_speech"; LM_DIR = os.path.join(REPO_DIR, "KenLM")
OUT_DIR  = f"{DRIVE_PROJECT}/compressed"; os.makedirs(OUT_DIR, exist_ok=True)
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/MatteoFasulo/silent_speech.git {REPO_DIR}
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("cwd:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
cwd: /content/silent_speech


In [10]:
import yaml
H5_REAL = f"{DRIVE_PROJECT}/emg_dataset.h5"
if not os.path.exists(H5_REAL):
    c = sorted(glob.glob("/content/drive/MyDrive/**/emg_dataset.h5", recursive=True)); assert c, "emg_dataset.h5 not found"
    H5_REAL = c[0]
for cfg in glob.glob(os.path.join(REPO_DIR, "config", "*.y*ml")):
    d = yaml.safe_load(open(cfg)) or {}
    if isinstance(d, dict) and "h5_path" in d:
        d["h5_path"] = H5_REAL; yaml.safe_dump(d, open(cfg, "w"), sort_keys=False)
print("h5:", H5_REAL)
os.makedirs(LM_DIR, exist_ok=True)
lm = glob.glob("/content/drive/MyDrive/**/lm.bin", recursive=True) + glob.glob(f"{REPO_DIR}/**/lm.bin", recursive=True)
assert lm, "lm.bin not found"
if os.path.abspath(lm[0]) != os.path.abspath(os.path.join(LM_DIR,"lm.bin")): shutil.copy(lm[0], os.path.join(LM_DIR,"lm.bin"))
lex = glob.glob(f"{REPO_DIR}/**/gaddy_derived_lexicon.txt", recursive=True); assert lex, "lexicon not found"
if os.path.abspath(lex[0]) != os.path.abspath(os.path.join(LM_DIR,"gaddy_derived_lexicon.txt")): shutil.copy(lex[0], os.path.join(LM_DIR,"gaddy_derived_lexicon.txt"))
print("KenLM dir:", os.listdir(LM_DIR))
if not os.path.exists(GADDY_CKPT):
    import requests
    print("fetching checkpoint from Zenodo 7183877 ...")
    rec = requests.get("https://zenodo.org/api/records/7183877").json()
    url = [f for f in rec["files"] if f["key"].endswith(".pt")][0]["links"]["self"]
    os.makedirs(os.path.dirname(GADDY_CKPT), exist_ok=True)
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(GADDY_CKPT,"wb") as fo:
            for ch in r.iter_content(1<<20): fo.write(ch)
print("checkpoint:", GADDY_CKPT, f"{os.path.getsize(GADDY_CKPT)/1e6:.1f} MB")

h5: /content/drive/MyDrive/silent_speech/emg_dataset.h5
KenLM dir: ['gaddy_derived_lexicon.txt', 'lm.bin']
checkpoint: /content/drive/MyDrive/KD/silent_speech/gaddy_recognition_model.pt 216.6 MB


In [11]:
# ===== Self-contained Gaddy recognition model (FIXED forward; keys identical to checkpoint) =====
import copy, random
import torch
from torch import nn
import torch.nn.functional as F
from typing import Optional

D_MODEL, N_HEAD, N_LAYERS, FFN, DROPOUT, RPD = 768, 8, 6, 3072, 0.0, 100

class LearnedRelativePositionalEmbedding(nn.Module):
    def __init__(self, max_relative_pos, num_heads, embedding_dim, unmasked=False,
                 heads_share_embeddings=False, add_to_values=False):
        super().__init__()
        self.max_relative_pos = max_relative_pos; self.num_heads = num_heads
        self.embedding_dim = embedding_dim; self.unmasked = unmasked
        self.heads_share_embeddings = heads_share_embeddings; self.add_to_values = add_to_values
        num_embeddings = 2 * max_relative_pos - 1 if unmasked else max_relative_pos
        embedding_size = [num_embeddings, embedding_dim, 1] if heads_share_embeddings \
            else [num_heads, num_embeddings, embedding_dim, 1]
        if add_to_values: embedding_size[-1] = 2
        initial_stddev = embedding_dim**(-0.5)
        self.embeddings = nn.Parameter(torch.zeros(*embedding_size))
        nn.init.normal_(self.embeddings, mean=0.0, std=initial_stddev)
    def forward(self, query, saved_state=None):
        length = query.shape[0]; decoder_step = False
        used = self.get_embeddings_for_query(length)
        vals = used[..., 1] if self.add_to_values else None
        pl = self.calculate_positional_logits(query, used[..., 0])
        pl = self.relative_to_absolute_indexing(pl, decoder_step)
        return (pl, vals)
    def get_embeddings_for_query(self, length):
        pad_length = max(length - self.max_relative_pos, 0)
        start_pos = max(self.max_relative_pos - length, 0)
        if self.unmasked:
            with torch.no_grad():
                padded = nn.functional.pad(self.embeddings, (0,0,0,0,pad_length,pad_length))
            used = padded.narrow(-3, start_pos, 2*length-1)
        else:
            with torch.no_grad():
                padded = nn.functional.pad(self.embeddings, (0,0,0,0,pad_length,0))
            used = padded.narrow(-3, start_pos, length)
        return used
    def calculate_positional_logits(self, query, relative_embeddings):
        if self.heads_share_embeddings:
            pl = torch.einsum("lbd,md->lbm", query, relative_embeddings)
        else:
            query = query.view(query.shape[0], -1, self.num_heads, self.embedding_dim)
            pl = torch.einsum("lbhd,hmd->lbhm", query, relative_embeddings)
            pl = pl.contiguous().view(pl.shape[0], -1, pl.shape[-1])
        length = query.size(0)
        if length > self.max_relative_pos:
            pad_length = length - self.max_relative_pos
            pl[:,:,:pad_length] -= 1e8
            if self.unmasked: pl[:,:,-pad_length:] -= 1e8
        return pl
    def relative_to_absolute_indexing(self, x, decoder_step):
        length, bsz_heads, _ = x.shape
        if decoder_step: return x.contiguous().view(bsz_heads, 1, -1)
        if self.unmasked:
            x = nn.functional.pad(x, (0,1)); x = x.transpose(0,1)
            x = x.contiguous().view(bsz_heads, length*2*length)
            x = nn.functional.pad(x, (0, length-1))
            x = x.view(bsz_heads, length+1, 2*length-1)
            return x[:, :length, length-1:]
        else:
            x = nn.functional.pad(x, (1,0)); x = x.transpose(0,1)
            x = x.contiguous().view(bsz_heads, length+1, length)
            return x[:, 1:, :]

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=256, n_head=4, dropout=0.1, relative_positional=True,
                 relative_positional_distance=100):
        super().__init__()
        self.d_model = d_model; self.n_head = n_head
        d_qkv = d_model // n_head
        assert d_qkv * n_head == d_model
        self.d_qkv = d_qkv
        self.w_q = nn.Parameter(torch.Tensor(n_head, d_model, d_qkv))
        self.w_k = nn.Parameter(torch.Tensor(n_head, d_model, d_qkv))
        self.w_v = nn.Parameter(torch.Tensor(n_head, d_model, d_qkv))
        self.w_o = nn.Parameter(torch.Tensor(n_head, d_qkv, d_model))
        nn.init.xavier_normal_(self.w_q); nn.init.xavier_normal_(self.w_k)
        nn.init.xavier_normal_(self.w_v); nn.init.xavier_normal_(self.w_o)
        self.dropout = nn.Dropout(dropout)
        if relative_positional:
            self.relative_positional = LearnedRelativePositionalEmbedding(
                relative_positional_distance, n_head, d_qkv, True)
        else:
            self.relative_positional = None
    def forward(self, x):
        q = torch.einsum('tbf,hfa->bhta', x, self.w_q)
        k = torch.einsum('tbf,hfa->bhta', x, self.w_k)
        v = torch.einsum('tbf,hfa->bhta', x, self.w_v)
        logits = torch.einsum('bhqa,bhka->bhqk', q, k) / (self.d_qkv ** 0.5)
        if self.relative_positional is not None:
            q_pos = q.permute(2,0,1,3)
            l,b,h,d = q_pos.size()
            position_logits, _ = self.relative_positional(q_pos.reshape(l,b*h,d))
            logits = logits + position_logits.view(b,h,l,l)
        probs = F.softmax(logits, dim=-1); probs = self.dropout(probs)
        o = torch.einsum('bhqk,bhka->bhqa', probs, v)
        out = torch.einsum('bhta,haf->tbf', o, self.w_o)
        return out

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward=2048, dropout=0.1,
                 relative_positional=True, relative_positional_distance=100):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, nhead, dropout=dropout,
            relative_positional=relative_positional, relative_positional_distance=relative_positional_distance)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout); self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU()
    def forward(self, src, src_mask=None, src_key_padding_mask=None, is_causal=False):
        src2 = self.self_attn(src)
        src = src + self.dropout1(src2); src = self.norm1(src)
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src = src + self.dropout2(src2); src = self.norm2(src)
        return src

class SimpleEncoder(nn.Module):
    """Key-identical, modern-torch-safe replacement for nn.TransformerEncoder."""
    def __init__(self, layer, n, norm=None):
        super().__init__()
        self.layers = nn.ModuleList([copy.deepcopy(layer) for _ in range(n)])
    def forward(self, x, mask=None, src_key_padding_mask=None, is_causal=None):
        for l in self.layers: x = l(x)
        return x

class ResBlock(nn.Module):
    def __init__(self, num_ins, num_outs, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(num_ins, num_outs, 3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm1d(num_outs)
        self.conv2 = nn.Conv1d(num_outs, num_outs, 3, padding=1)
        self.bn2 = nn.BatchNorm1d(num_outs)
        if stride != 1 or num_ins != num_outs:
            self.residual_path = nn.Conv1d(num_ins, num_outs, 1, stride=stride)
            self.res_norm = nn.BatchNorm1d(num_outs)
        else:
            self.residual_path = None
    def forward(self, x):
        iv = x
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        res = self.res_norm(self.residual_path(iv)) if self.residual_path is not None else iv
        return F.relu(x + res)

class Model(nn.Module):
    def __init__(self, num_features, num_outs, num_aux_outs=None,
                 model_size=D_MODEL, num_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.conv_blocks = nn.Sequential(
            ResBlock(8, model_size, 2),
            ResBlock(model_size, model_size, 2),
            ResBlock(model_size, model_size, 2),
        )
        self.w_raw_in = nn.Linear(model_size, model_size)
        layer = TransformerEncoderLayer(d_model=model_size, nhead=N_HEAD, relative_positional=True,
            relative_positional_distance=RPD, dim_feedforward=FFN, dropout=dropout)
        self.transformer = SimpleEncoder(layer, num_layers)
        self.w_out = nn.Linear(model_size, num_outs)
        self.has_aux_out = num_aux_outs is not None
        if self.has_aux_out:
            self.w_aux = nn.Linear(model_size, num_aux_outs)
    def forward(self, x_feat, x_raw, session_ids):
        if self.training:
            r = random.randrange(8)
            if r > 0:
                shifted = torch.zeros_like(x_raw)
                shifted[:, :-r, :] = x_raw[:, r:, :]
                x_raw = shifted
        x_raw = x_raw.transpose(1,2)
        x_raw = self.conv_blocks(x_raw)
        x_raw = x_raw.transpose(1,2)
        x_raw = self.w_raw_in(x_raw)
        x = x_raw.transpose(0,1)
        x = self.transformer(x)
        x = x.transpose(0,1)
        if self.has_aux_out:
            return self.w_out(x), self.w_aux(x)
        return self.w_out(x)


# ---- build + load teacher ----
import os
from hdf5_dataset import H5EmgDataset
_probe = H5EmgDataset(dev=False, test=True)
N_CHARS = len(_probe.text_transform.chars); NUM_OUTS = N_CHARS + 1
print("n_chars:", N_CHARS, "| num_outs:", NUM_OUTS)
model = Model(num_features=0, num_outs=NUM_OUTS).eval()
sd = torch.load(GADDY_CKPT, map_location="cpu", weights_only=False)
if isinstance(sd, dict) and "model_state_dict" in sd: sd = sd["model_state_dict"]
missing, unexpected = model.load_state_dict(sd, strict=False)
assert len(missing)==0, ("checkpoint mismatch:", missing[:8])
print("teacher params:", sum(p.numel() for p in model.parameters()))

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

n_chars: 37 | num_outs: 38
teacher params: 54117926


In [12]:
import numpy as np, tqdm, jiwer, copy
import torch.nn.functional as F
from torchaudio.models.decoder import ctc_decoder

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(os.cpu_count() or 4)

trainset = H5EmgDataset(dev=False, test=False)
devset   = H5EmgDataset(dev=True,  test=False)
testset  = _probe
TT = testset.text_transform; CHARS = TT.chars; BLANK = len(CHARS)
print("train/dev/test:", len(trainset), len(devset), len(testset), "| blank id:", BLANK)

def cache_set(ds, limit=None):
    out=[]
    for i in range(len(ds)):
        ex=ds[i]; t=TT.clean_text(ex["text"])
        if t!="": out.append((ex["raw_emg"], t))
        if limit and len(out)>=limit: break
    return out
DEV   = cache_set(devset, limit=300)   # dev subset for FAST early-stopping signal
TESTC = cache_set(testset)             # full silent test set for the final number
print("dev(early-stop) :", len(DEV), "| test:", len(TESTC))

# FAST greedy WER (no LM) -> drives early stopping
def greedy_wer(m, data, device=DEVICE):
    m.eval(); dummy=torch.zeros(1,1,dtype=torch.long,device=device); refs,preds=[],[]
    with torch.no_grad():
        for raw,txt in data:
            x=raw.unsqueeze(0).to(device); ids=m(x,x,dummy).argmax(-1)[0].tolist()
            out=[]; prev=-1
            for i in ids:
                if i!=prev and i!=BLANK: out.append(i)
                prev=i
            preds.append(TT.clean_text("".join(CHARS[i] for i in out)) or " "); refs.append(txt)
    m.train(); return jiwer.wer(refs,preds)

# ACCURATE KenLM beam WER -> final test number only
def build_decoder(beam=1500, lm_weight=4.0, word_score=-1.75):
    tkns=[c for c in CHARS]+["_"]
    return ctc_decoder(lexicon=os.path.join(LM_DIR,"gaddy_derived_lexicon.txt"), tokens=tkns,
        lm=os.path.join(LM_DIR,"lm.bin"), blank_token="_", sil_token="|", nbest=1,
        lm_weight=lm_weight, word_score=word_score, sil_score=0.0, beam_size=beam)
def beam_wer(m, data, decoder, device=DEVICE, desc="test"):
    m.eval(); dummy=torch.zeros(1,1,dtype=torch.long,device=device); refs,preds=[],[]
    with torch.no_grad():
        for raw,txt in tqdm.tqdm(data, desc=desc):
            x=raw.unsqueeze(0).to(device); logp=F.log_softmax(m(x,x,dummy).float().cpu(),-1)
            preds.append(TT.clean_text(" ".join(decoder(logp)[0][0].words).strip())); refs.append(txt)
    return jiwer.wer(refs,preds)
def sd_mb(m): torch.save(m.state_dict(),"/content/_t.pt"); return os.path.getsize("/content/_t.pt")/1e6

# teacher logits cached in RAM (temperature-scaled)
TEMP = 2.0
teacher = copy.deepcopy(model).eval().to(DEVICE)
for p in teacher.parameters(): p.requires_grad_(False)
print("caching teacher logits for", len(trainset), "train utterances ...")
CACHE=[]; dummy=torch.zeros(1,1,dtype=torch.long,device=DEVICE)
with torch.no_grad():
    for i in tqdm.tqdm(range(len(trainset))):
        ex=trainset[i]; tgt=TT.text_to_int(ex["text"])
        if len(tgt)==0: continue
        raw=ex["raw_emg"].unsqueeze(0).to(DEVICE)
        tl=F.log_softmax(teacher(raw,raw,dummy)/TEMP,-1).cpu()
        CACHE.append((ex["raw_emg"], tl, torch.tensor(tgt)))
print("cached teacher logits:", len(CACHE))

train/dev/test: 1289 200 99 | blank id: 37
dev(early-stop) : 199 | test: 98
caching teacher logits for 1289 train utterances ...


100%|██████████| 1289/1289 [00:36<00:00, 35.76it/s]

cached teacher logits: 1285


In [15]:
# ===== KD TRAINING (fixed): early-stop on dev KenLM-BEAM WER, not greedy =====
STUDENT_SIZE, STUDENT_LAYERS = 256, 3
MAX_EPOCHS, EVAL_EVERY, PATIENCE = 200, 5, 6      # eval every 5 epochs; stop after 6 evals (=30 ep) with no gain
ALPHA, LR, CLIP = 0.7, 3e-4, 5.0                  # same ALPHA as your old run, for a fair comparison

dev_decoder = build_decoder(beam=500)             # LM decoder for dev monitoring (smaller beam = faster)
DEV_EVAL    = DEV[:60]                             # small dev subset so beam eval stays quick (~30-60s)
def dev_wer(m): return beam_wer(m, DEV_EVAL, dev_decoder, desc="dev(beam)")

student = Model(0, NUM_OUTS, model_size=STUDENT_SIZE, num_layers=STUDENT_LAYERS).train().to(DEVICE)
print("student params:", sum(p.numel() for p in student.parameters()))
opt   = torch.optim.AdamW(student.parameters(), lr=LR)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=2)
dummy = torch.zeros(1,1,dtype=torch.long,device=DEVICE)
best=1e9; best_state=None; bad=0
for ep in range(1, MAX_EPOCHS+1):
    student.train(); tot=0.0
    for idx in torch.randperm(len(CACHE)):
        raw, tl, tgt = CACHE[int(idx)]
        raw=raw.unsqueeze(0).to(DEVICE); tl=tl.to(DEVICE); tgt=tgt.to(DEVICE)
        s=student(raw,raw,dummy)
        kd =F.kl_div(F.log_softmax(s/TEMP,-1), tl, log_target=True, reduction="batchmean")*(TEMP*TEMP)
        ctc=F.ctc_loss(F.log_softmax(s,-1).transpose(0,1), tgt,
                       torch.tensor([s.shape[1]],device=DEVICE), torch.tensor([tgt.numel()],device=DEVICE),
                       blank=BLANK, zero_infinity=True)
        loss=ALPHA*kd+(1-ALPHA)*ctc
        opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(student.parameters(), CLIP); opt.step()
        tot+=loss.item()
    if ep % EVAL_EVERY == 0:
        dw = dev_wer(student); sched.step(dw)
        if dw < best - 1e-4:
            best=dw; best_state={k:v.cpu().clone() for k,v in student.state_dict().items()}; bad=0; flag="*best"
        else:
            bad+=1; flag=f"no-improve {bad}/{PATIENCE}"
        print(f"epoch {ep:3d}  loss {tot/len(CACHE):.2f}  dev_BEAM_WER {dw*100:5.1f}%  lr {opt.param_groups[0]['lr']:.1e}  {flag}")
        if bad>=PATIENCE: print(f"early stop at epoch {ep} (best dev beam WER {best*100:.1f}%)"); break
if best_state is not None: student.load_state_dict(best_state)
BEST_KD=os.path.join(OUT_DIR,"gaddy_kd_student.pt"); torch.save(student.state_dict(),BEST_KD)
print("best student saved ->", BEST_KD, f"| best dev beam WER {best*100:.1f}%")

student params: 6875686


dev(beam): 100%|██████████| 60/60 [01:03<00:00,  1.07s/it]


epoch   5  loss 1301.91  dev_BEAM_WER  86.0%  lr 3.0e-04  *best


dev(beam): 100%|██████████| 60/60 [00:59<00:00,  1.00it/s]


epoch  10  loss 921.08  dev_BEAM_WER  72.3%  lr 3.0e-04  *best


...

epoch  55  loss 189.86  dev_BEAM_WER  67.3%  lr 1.5e-04  no-improve 4/6


dev(beam): 100%|██████████| 60/60 [00:48<00:00,  1.24it/s]


epoch  60  loss 164.25  dev_BEAM_WER  66.7%  lr 1.5e-04  no-improve 5/6


dev(beam): 100%|██████████| 60/60 [00:48<00:00,  1.22it/s]

epoch  65  loss 161.91  dev_BEAM_WER  66.7%  lr 7.5e-05  no-improve 6/6
early stop at epoch 65 (best dev beam WER 66.3%)
best student saved -> /content/drive/MyDrive/KD/silent_speech/compressed/gaddy_kd_student.pt | best dev beam WER 66.3%


In [16]:
# ===== FINAL (device-safe): KenLM beam WER on the silent TEST set — student vs teacher =====
import pandas as pd, copy, os

decoder = build_decoder(beam=1500)

# --- KD student ---
student = student.to(DEVICE).eval()
kd_wer  = beam_wer(student, TESTC, decoder, device=DEVICE, desc="KD student (test)")
kd_size = sd_mb(student)
print(f"\nKD student : test WER {kd_wer*100:.2f}%  | size {kd_size:.2f} MB  | params {sum(p.numel() for p in student.parameters())}")

# --- teacher (move to GPU too — this was the crash) ---
teacher_eval = copy.deepcopy(model).to(DEVICE).eval()
tw      = beam_wer(teacher_eval, TESTC, decoder, device=DEVICE, desc="teacher 54M (test)")
tw_size = sd_mb(teacher_eval)
print(f"Teacher 54M: test WER {tw*100:.2f}%  | size {tw_size:.2f} MB")
print(f"KD vs teacher: {kd_wer*100:.2f}% vs {tw*100:.2f}%  -> within 10%? {'YES' if kd_wer <= tw*1.10 else 'NO'}")

# --- save + print CSV ---
rows = [{"model":"KD student","WER%":round(kd_wer*100,2),"size_MB":round(kd_size,2)},
        {"model":"Teacher 54M","WER%":round(tw*100,2),"size_MB":round(tw_size,2)}]
CSV = f"{DRIVE_PROJECT}/gaddy_kd_result.csv"
pd.DataFrame(rows).to_csv(CSV, index=False)
print("\nsaved ->", CSV)
print(pd.read_csv(CSV).to_string(index=False))

KD student (test): 100%|██████████| 98/98 [03:32<00:00,  2.17s/it]



KD student : test WER 63.30%  | size 27.55 MB  | params 6875686


teacher 54M (test): 100%|██████████| 98/98 [02:23<00:00,  1.47s/it]


Teacher 54M: test WER 21.63%  | size 216.57 MB
KD vs teacher: 63.30% vs 21.63%  -> within 10%? NO

saved -> /content/drive/MyDrive/KD/silent_speech/gaddy_kd_result.csv
      model  WER%  size_MB
 KD student 63.30    27.55
Teacher 54M 21.63   216.57
